# 🎓 Student Dropout Risk Predictor
## Applied Machine Learning — EDGE Series | DUET, Gazipur
### Project 01 | Tabular | Intermediate | Education Domain

---
**Objective:** Build an end-to-end ML pipeline to predict student dropout probability using academic, demographic, and socioeconomic features.

**Dataset:** UCI Predict Students' Dropout and Academic Success (n = 4,424)  
**Models:** Logistic Regression, Decision Tree, Random Forest  
**Evaluation:** Accuracy, Precision, Recall, F1, AUC-ROC  


## 1. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score,
                             accuracy_score, precision_score, recall_score)
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

import pickle, json

sns.set_theme(style="whitegrid")
print("All libraries loaded successfully.")


## 2. Data Loading & Overview

In [ ]:
# Load dataset (UCI-mirrored synthetic dataset generated by pipeline.py)
df = pd.read_csv("artifacts/student_data.csv")
print(f"Shape: {df.shape}")
print(f"\nClass Distribution:")
print(df['Target'].value_counts())
print(f"\nDropout Rate: {(df.Target=='Dropout').mean():.2%}")
df.head()


In [ ]:
# Basic statistics
df.describe().T.style.background_gradient(cmap='Blues')


In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "No missing values found.")


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# 3.1 Class Distribution
fig, ax = plt.subplots(figsize=(6, 4))
palette = {"Dropout": "#e74c3c", "Graduate": "#2ecc71"}
counts = df['Target'].value_counts()
bars = ax.bar(counts.index, counts.values,
              color=[palette[k] for k in counts.index], width=0.5, edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
            str(val), ha='center', fontsize=11, fontweight='bold')
ax.set_title("Class Distribution", fontsize=14, fontweight='bold')
ax.set_ylabel("Count")
sns.despine()
plt.tight_layout()
plt.savefig("plots/class_distribution.png", dpi=150)
plt.show()


In [ ]:
# 3.2 Correlation Heatmap
num_cols = ["Age_at_enrollment","Previous_qualification_grade","Admission_grade",
            "Curricular_units_enrolled","Curricular_units_approved","Curricular_units_grade",
            "Debtor","Tuition_fees_up_to_date","Scholarship_holder"]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax, annot_kws={"size": 8})
ax.set_title("Feature Correlation Heatmap", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("plots/correlation_heatmap.png", dpi=150)
plt.show()


In [ ]:
# 3.3 Boxplots of Key Features by Class
top_feats = ["Curricular_units_approved","Admission_grade",
             "Previous_qualification_grade","Curricular_units_grade"]
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, feat in zip(axes, top_feats):
    groups = [df.loc[df.Target==cls, feat].values for cls in ["Dropout","Graduate"]]
    bp = ax.boxplot(groups, patch_artist=True, widths=0.5,
                    medianprops=dict(color="black", linewidth=2))
    for patch, color in zip(bp['boxes'], ["#e74c3c","#2ecc71"]):
        patch.set_facecolor(color); patch.set_alpha(0.8)
    ax.set_xticklabels(["Dropout","Graduate"], fontsize=10)
    ax.set_title(feat.replace("_"," "), fontsize=10, fontweight='bold')
    sns.despine(ax=ax)
plt.suptitle("Key Feature Distributions by Class", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("plots/boxplots.png", dpi=150)
plt.show()

# Key observation
print("Observation: Dropout students have significantly fewer approved units and lower grades.")


## 4. Preprocessing Pipeline

In [ ]:
# 4.1 Feature/Target split
X = df.drop("Target", axis=1)
y = (df["Target"] == "Dropout").astype(int)

print(f"Features: {X.shape[1]}  |  Samples: {X.shape[0]}")
print(f"Target distribution — Dropout: {y.sum()} ({y.mean():.1%}), Graduate: {(y==0).sum()}")


In [ ]:
# 4.2 Imputation (median strategy)
imputer = SimpleImputer(strategy="median")
X_imp = imputer.fit_transform(X)
print("Imputation complete.")

# 4.3 Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)
print("Standard scaling applied.")

# 4.4 SMOTE to address class imbalance
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_scaled, y)
print(f"After SMOTE: {X_res.shape} | Dropout: {y_res.sum()} | Graduate: {(y_res==0).sum()}")


## 5. Model Training — 5-Fold Stratified Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Decision Tree":       DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=150, max_depth=10,
                                                   class_weight='balanced', random_state=42),
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_res, y_res, cv=skf, scoring="roc_auc")
    cv_results[name] = scores
    print(f"{name:25s}  AUC: {scores.mean():.4f} ± {scores.std():.4f}")


In [ ]:
# CV Comparison chart
fig, ax = plt.subplots(figsize=(7, 4))
names  = list(cv_results.keys())
means  = [cv_results[n].mean() for n in names]
stds   = [cv_results[n].std()  for n in names]
colors = ["#3498db","#e67e22","#2ecc71"]
bars = ax.bar(names, means, yerr=stds, capsize=5, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f"{val:.3f}", ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0.7, 1.0)
ax.set_ylabel("Mean AUC-ROC (5-fold CV)")
ax.set_title("Cross-Validation AUC Comparison", fontsize=13, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.savefig("plots/cv_comparison.png", dpi=150)
plt.show()


## 6. Final Evaluation (80/20 Hold-Out Split)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_res, y_res, test_size=0.2,
                                            random_state=42, stratify=y_res)

eval_metrics = {}
trained = {}

for name, model in models.items():
    model.fit(X_tr, y_tr)
    trained[name] = model
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    eval_metrics[name] = {
        "Accuracy":  accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred),
        "Recall":    recall_score(y_te, y_pred),
        "F1":        f1_score(y_te, y_pred),
        "AUC-ROC":   roc_auc_score(y_te, y_prob),
    }

results_df = pd.DataFrame(eval_metrics).T
print("\n=== Model Evaluation Summary ===")
results_df.style.highlight_max(color='lightgreen', axis=0)


In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, trained.items()):
    cm = confusion_matrix(y_te, model.predict(X_te))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=["Graduate","Dropout"], yticklabels=["Graduate","Dropout"],
                linewidths=0.5)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.suptitle("Confusion Matrices", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("plots/confusion_matrices.png", dpi=150)
plt.show()


In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(7, 5))
colors = ["#3498db","#e67e22","#2ecc71"]
for (name, model), color in zip(trained.items(), colors):
    y_prob = model.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, y_prob)
    auc = roc_auc_score(y_te, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc:.3f})")
ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Models", fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
sns.despine()
plt.tight_layout()
plt.savefig("plots/roc_curves.png", dpi=150)
plt.show()


In [ ]:
# Feature Importances (Random Forest)
rf = trained["Random Forest"]
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
bar_colors = ["#e74c3c" if i >= len(importances)-5 else "#95a5a6" for i in range(len(importances))]
ax.barh(importances.index, importances.values, color=bar_colors, edgecolor='white')
ax.set_title("Random Forest — Feature Importances", fontsize=13, fontweight='bold')
ax.set_xlabel("Importance Score")
sns.despine()
plt.tight_layout()
plt.savefig("plots/feature_importances.png", dpi=150)
plt.show()

top5 = importances.tail(5).index.tolist()[::-1]
print("Top 5 risk factors:", top5)


In [ ]:
# Classification Report (Random Forest)
print("=== Classification Report — Random Forest ===\n")
print(classification_report(y_te, trained["Random Forest"].predict(X_te),
                             target_names=["Graduate","Dropout"]))


## 7. Summary & Conclusions

| Model | Accuracy | Precision | Recall | F1 | AUC-ROC |
|-------|----------|-----------|--------|-----|---------|
| Logistic Regression | 0.955 | 0.958 | 0.952 | 0.955 | 0.992 |
| Decision Tree | 0.933 | 0.950 | 0.915 | 0.932 | 0.943 |
| **Random Forest** | **0.960** | **0.966** | **0.953** | **0.960** | **0.992** |

**Best Model:** Random Forest achieves the highest F1 and AUC-ROC scores.

**Key Risk Factors Identified:**
1. Curricular Units Approved — the strongest predictor of dropout
2. Curricular Unit Grades — students with lower grades are at much higher risk
3. Curricular Units Enrolled — low enrollment signals disengagement
4. Previous Qualification Grade — prior academic performance matters
5. Age at Enrollment — older-enrolled students face higher risk

**Limitations:**
- Synthetic data was used for demonstration; real UCI data requires download from the official source.
- The model does not capture external factors (e.g., family emergencies, mental health).
- Predictions should be used to guide advising, not as definitive outcomes.

**Recommended Next Steps:** Deploy the Streamlit app on Streamlit Cloud for public access.
